# 71 — Parent Document Retriever
## Small Chunks to Find, Full Docs to Answer
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/71-parent-document-retriever/parent_document_retriever_workbook.ipynb)

Every RAG system faces a cruel tradeoff: **small chunks match queries precisely** but contain too little context for the LLM to answer well. **Large chunks give rich context** but are so long that their embeddings become diffuse, dragging unrelated content into the top-k results. The Parent Document Retriever solves this by maintaining **two representations of the same content** — tiny child chunks for search, full parent documents for generation. When a child chunk wins the similarity race, the retriever transparently swaps it out for its full parent before passing context to the LLM.

This workshop builds the pattern from first principles using LangChain's `ParentDocumentRetriever`, then wires it into a two-node LangGraph pipeline.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — The precision-vs-context tradeoff explained |
| 2 | **Setup** — Install deps, API key, imports |
| 3 | **Sample Corpus** — The three documents we will index |
| 4 | **Splitting Strategy** — Child vs parent chunk sizes |
| 5 | **Vectorstore & Docstore** — Where child indexes and parent docs live |
| 6 | **Building the Retriever** — `ParentDocumentRetriever` constructor |
| 7 | **Indexing Documents** — What happens inside `add_documents()` |
| 8 | **Child Search Demo** — What raw similarity search returns |
| 9 | **Parent Retrieval Demo** — The swap in action |
| 10 | **Size Comparison** — Quantifying the precision-vs-context gap |
| 11 | **LangGraph State** — Defining `ParentDocState` |
| 12 | **Retrieve Node** — The retrieval step in the graph |
| 13 | **Generate Node** — Prompting the LLM with parent context |
| 14 | **Graph Assembly** — Wiring START → retrieve_parent → generate → END |
| 15 | **Full Pipeline Run** — All three questions end-to-end |
| 16 | **Answer Quality Comparison** — Child-context vs parent-context answers |
| 17 | **Chunk Size Sensitivity** — What happens when you change CHILD_CHUNK_SIZE |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `langchain`, `langchain-openai`, `langchain-chroma`, `langchain-text-splitters`, `chromadb`, `langgraph`, `python-dotenv`

### Key References
> - [LangChain ParentDocumentRetriever docs](https://python.langchain.com/docs/modules/data_connection/retrievers/parent_document_retriever)
> - [LangGraph StateGraph docs](https://langchain-ai.github.io/langgraph/reference/graphs/)
> - Original RAG paper: Lewis et al. 2020 — [arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401)

## Part 1 — The Core Tradeoff

Before writing a single line of code, understand the problem the Parent Document Retriever exists to solve.

### Why chunking matters for RAG

When you embed a document, you compress its meaning into a fixed-size vector (e.g. 1536 floats for `text-embedding-3-small`). That vector captures the **average semantic content** of the text. This has an important consequence:

```
SHORT TEXT:  "multi-head attention"
  → embedding tightly focused on ONE concept
  → precise cosine similarity to similar queries

LONG TEXT:   "Transformers use attention ... BERT ... GPT ... scale ..."
  → embedding is an average of MANY concepts
  → similarity to any single query is diluted
```

### The tradeoff table

| Strategy | Embedding Precision | Context for LLM | Problem |
|----------|--------------------|-----------------|---------|
| Tiny chunks (100 chars) | ✅ High | ❌ Too thin | LLM lacks context to answer |
| Large chunks (1000 chars) | ❌ Diluted | ✅ Rich | Wrong chunks retrieved |
| **Parent Document Retriever** | **✅ High** | **✅ Rich** | **Best of both worlds** |

### How the Parent Document Retriever solves it

```
INDEXING TIME:
  Full doc → parent_splitter → [parent_1, parent_2, ...]
                               each parent → child_splitter → [child_a, child_b, ...]
                               child embeddings → vectorstore
                               parent text → docstore (keyed by parent_id)
                               child metadata → { parent_id: "..." }

RETRIEVAL TIME:
  query → embed → vectorstore similarity_search → top child chunks
                → look up parent_id from child metadata
                → fetch full parent from docstore
                → return parent docs (not child chunks!) to LLM
```

The key insight: the **vectorstore never stores parent text**. It stores child embeddings. The **docstore never serves retrieval** — it only serves lookup by key. The two stores do completely different jobs.

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-chroma",
        "langchain-text-splitters",
        "chromadb",
        "langgraph",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from typing import TypedDict

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

print("Imports OK")

## Part 3 — Sample Corpus

We'll work with three dense technical paragraphs about AI fundamentals. Each is roughly 500-600 characters — long enough to contain multiple ideas, short enough to read in the notebook.

These represent the kind of content you'd get from chunked PDFs, technical docs, or knowledge-base articles in a real RAG application.

In [ ]:
SAMPLE_DOCS = [
    """The transformer architecture revolutionized natural language processing. Introduced in 2017 by Vaswani et al., it replaced recurrent networks with self-attention mechanisms. This allowed parallel computation and better capture of long-range dependencies. The key innovation was multi-head attention, which lets the model focus on different parts of the input simultaneously. Transformers became the foundation for BERT, GPT, and all modern large language models. Their ability to scale with more data and computation made them the dominant architecture in AI today.""",

    """Retrieval-augmented generation (RAG) combines the strengths of retrieval systems and generative models. Instead of relying solely on parametric knowledge stored in model weights, RAG retrieves relevant documents from an external knowledge base at inference time. This approach reduces hallucination and allows models to access up-to-date information. The typical RAG pipeline has three stages: document indexing, retrieval, and generation. Vector embeddings enable semantic search, where queries and documents are matched by meaning rather than exact keywords. RAG has become the standard architecture for building AI applications over private or frequently updated data.""",

    """Neural network training involves adjusting millions of parameters to minimize a loss function. The primary algorithm is stochastic gradient descent (SGD) with backpropagation. In each training iteration, the model makes a forward pass to compute predictions, then a backward pass to compute gradients. The optimizer updates weights based on those gradients. Modern optimizers like Adam adapt the learning rate for each parameter. Regularization techniques like dropout prevent overfitting by randomly disabling neurons during training. Training large models requires distributed computing across multiple GPUs or TPUs.""",
]

SAMPLE_QUESTIONS = [
    "What was the key innovation in the transformer architecture?",
    "How does RAG reduce hallucination?",
    "What is stochastic gradient descent?",
]

for i, doc in enumerate(SAMPLE_DOCS):
    print(f"Doc {i+1}: {len(doc)} chars — {doc[:60]}...")

## Part 4 — Splitting Strategy

The core parameters that control the precision-vs-context balance are the **chunk sizes**.

```
CHILD_CHUNK_SIZE = 100   # ~15-20 words — embeds a single tight idea
PARENT_CHUNK_SIZE = 600  # ~90-100 words — full topic coverage
```

Both splitters use `RecursiveCharacterTextSplitter`, which tries to split on natural boundaries (paragraphs, sentences, words) before falling back to hard character splits.

### Why these specific values?

| Parameter | Our value | Typical range | Too small means | Too large means |
|-----------|-----------|---------------|-----------------|-----------------|
| `CHILD_CHUNK_SIZE` | 100 | 50–200 chars | No semantic content | Diluted embedding |
| `child_overlap` | 10 | 5–20% of chunk | Missing boundary context | Duplicate retrieval |
| `PARENT_CHUNK_SIZE` | 600 | 400–2000 chars | LLM lacks context | Irrelevant text included |
| `parent_overlap` | 50 | 5–10% of chunk | Missing boundary context | Token waste |

**Rule of thumb**: child size should be 5-10x smaller than parent size. At our ratio (6x), each parent will contain roughly 4-6 child chunks.

In [ ]:
CHILD_CHUNK_SIZE = 100
PARENT_CHUNK_SIZE = 600

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=10,
)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=50,
)

# Preview how the first document splits at each level
sample_doc = Document(page_content=SAMPLE_DOCS[0])

parent_splits = parent_splitter.split_documents([sample_doc])
child_splits = child_splitter.split_documents([sample_doc])

print(f"Doc 1 ({len(SAMPLE_DOCS[0])} chars)")
print(f"  Parent splits: {len(parent_splits)}")
for i, p in enumerate(parent_splits):
    print(f"    Parent {i}: {len(p.page_content)} chars — {p.page_content[:60]}...")
print(f"  Child splits:  {len(child_splits)}")
for i, c in enumerate(child_splits):
    print(f"    Child {i}:  {len(c.page_content)} chars — {c.page_content[:50]}...")

## Part 5 — Vectorstore and Docstore

The Parent Document Retriever uses **two separate storage systems** with completely different responsibilities:

```
┌─────────────────────────────────────────────────────────┐
│  VECTORSTORE (Chroma)                                   │
│  • Stores: child chunk EMBEDDINGS                       │
│  • Keys: auto-generated child chunk IDs                 │
│  • Metadata: { "doc_id": "<parent_id>" }               │
│  • Used for: similarity search at query time            │
│  • Persists: to disk (or in-memory for this example)    │
└─────────────────────────────────────────────────────────┘
                         ↕  (parent_id links the two)
┌─────────────────────────────────────────────────────────┐
│  DOCSTORE (InMemoryStore)                               │
│  • Stores: parent chunk TEXT                            │
│  • Keys: parent document IDs                            │
│  • Used for: key-value lookup after child match         │
│  • Persists: in RAM only (lost on restart)              │
└─────────────────────────────────────────────────────────┘
```

**Production note**: `InMemoryStore` is fine for prototyping, but in production you'd use a persistent docstore — e.g. a Redis-backed store or a database-backed store — so parent docs survive restarts.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Chroma stores child chunk embeddings.
# collection_name scopes this index — important if you reuse Chroma across notebooks.
vectorstore = Chroma(
    collection_name="parent-doc-children",
    embedding_function=embeddings,
)

# InMemoryStore is a simple dict-like key-value store.
# It lives in-process and is lost when the kernel restarts.
docstore = InMemoryStore()

print("Vectorstore collection:", vectorstore._collection.name)
print("Docstore type:", type(docstore).__name__)

## Part 6 — Building the Retriever

The `ParentDocumentRetriever` is the glue that connects the vectorstore, docstore, and both splitters. Its constructor wires them together — you don't call either splitter directly at query time.

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

print("Retriever type:", type(retriever).__name__)
print("Child splitter chunk size:", retriever.child_splitter._chunk_size)
print("Parent splitter chunk size:", retriever.parent_splitter._chunk_size)

## Part 7 — Indexing Documents

When you call `retriever.add_documents(docs)`, this is what happens internally:

```
for each doc in docs:
    1. parent_splitter.split(doc)  → [parent_chunk_1, parent_chunk_2, ...]
    2. for each parent_chunk:
          assign a unique parent_id
          docstore.mset([(parent_id, parent_chunk)])   # store parent text
          child_splitter.split(parent_chunk) → [child_1, child_2, ...]
          for each child:
              child.metadata["doc_id"] = parent_id     # link child → parent
          vectorstore.add_documents([child_1, child_2, ...])  # embed and store
```

The docstore holds the source of truth. The vectorstore holds only the index.

In [ ]:
docs = [Document(page_content=d) for d in SAMPLE_DOCS]

# This call:
#   - splits each doc into parent chunks with parent_splitter
#   - stores parent chunks in docstore
#   - splits each parent chunk into child chunks with child_splitter
#   - embeds children and stores them in vectorstore with parent_id metadata
retriever.add_documents(docs)

# How many child chunks ended up in the vectorstore?
child_count = vectorstore._collection.count()
print(f"Child chunks in vectorstore: {child_count}")

# Inspect one child chunk to see the parent_id link in metadata
sample_children = vectorstore.similarity_search("transformer attention", k=2)
for c in sample_children:
    print(f"  Child text: {c.page_content[:60]}...")
    print(f"  Metadata:   {c.metadata}")

## Part 8 — Child Search Demo

Let's first look at what **raw similarity search** returns — the tiny child chunks that win the embedding race. This is what a naive RAG system would pass to the LLM.

In [ ]:
question = SAMPLE_QUESTIONS[0]  # "What was the key innovation in the transformer architecture?"

child_results = vectorstore.similarity_search(question, k=4)

print(f"Query: {question}")
print(f"\nTop {len(child_results)} child chunks from vectorstore.similarity_search():")
print("-" * 60)
for i, chunk in enumerate(child_results):
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"  {chunk.page_content}")
    print(f"  parent_id: {chunk.metadata.get('doc_id', 'n/a')}")

total_child_chars = sum(len(c.page_content) for c in child_results)
print(f"\nTotal context if we used child chunks: {total_child_chars} chars")

## Part 9 — Parent Retrieval Demo

Now call `retriever.invoke()` — this triggers the full small-to-large swap. Same query, but the output is the **full parent documents** that contained the matching child chunks.

In [ ]:
parent_results = retriever.invoke(question)

print(f"Query: {question}")
print(f"\nParent documents from retriever.invoke():")
print("-" * 60)
for i, doc in enumerate(parent_results):
    print(f"\nParent {i+1} ({len(doc.page_content)} chars):")
    print(f"  {doc.page_content[:200]}...")

total_parent_chars = sum(len(d.page_content) for d in parent_results)
print(f"\nTotal context if we use parent docs: {total_parent_chars} chars")

## Part 10 — Size Comparison

Let's quantify the difference across all three questions. This is the clearest way to see the tradeoff in action.

In [ ]:
print(f"{'Question':<50} {'Child chars':>12} {'Parent chars':>13} {'Ratio':>6}")
print("-" * 85)

for q in SAMPLE_QUESTIONS:
    children = vectorstore.similarity_search(q, k=4)
    parents = retriever.invoke(q)

    child_total = sum(len(c.page_content) for c in children)
    parent_total = sum(len(p.page_content) for p in parents)
    ratio = parent_total / max(child_total, 1)

    print(f"{q[:48]:<50} {child_total:>12,} {parent_total:>13,} {ratio:>5.1f}x")

print()
print("The parent docs provide 4-8x more context than raw child chunks.")
print("This is the content the LLM needs to write a complete, accurate answer.")

## Part 11 — LangGraph State

Now we wire the retriever into a LangGraph pipeline. The first step is defining the **state schema** — the data that flows between nodes.

```
ParentDocState
├── question    : str           — the user's query
├── child_chunks: list[str]     — raw similarity results (for inspection)
├── parent_docs : list[str]     — full parent documents (fed to LLM)
└── answer      : str           — LLM-generated answer
```

We use `TypedDict` rather than Pydantic `BaseModel` because:
- No runtime validation overhead (LangGraph validates structure at compile time)
- Lighter import footprint
- `TypedDict` is idiomatic for LangGraph state when you don't need custom validators

In [ ]:
class ParentDocState(TypedDict):
    question: str
    child_chunks: list[str]
    parent_docs: list[str]
    answer: str

# Verify the initial state we'll pass at invocation time
initial_state: ParentDocState = {
    "question": "example question",
    "child_chunks": [],
    "parent_docs": [],
    "answer": "",
}
print("State schema keys:", list(initial_state.keys()))
print("Initial state:", initial_state)

## Part 12 — Retrieve Node

The retrieve node runs both lookups:
1. `retriever.invoke()` — returns parent docs for the LLM
2. `vectorstore.similarity_search()` — returns child chunks so we can inspect them

We store both in state so downstream nodes (or inspection code) can compare them.

In [ ]:
def retrieve_parent(state: ParentDocState) -> dict:
    """Retrieve parent documents (for LLM context) and child chunks (for inspection)."""
    parent_docs = retriever.invoke(state["question"])
    child_docs = vectorstore.similarity_search(state["question"], k=4)
    return {
        "parent_docs": [d.page_content for d in parent_docs],
        "child_chunks": [d.page_content for d in child_docs],
    }

# Test the node in isolation before adding to the graph
test_state: ParentDocState = {
    "question": SAMPLE_QUESTIONS[1],
    "child_chunks": [],
    "parent_docs": [],
    "answer": "",
}
retrieved = retrieve_parent(test_state)
print(f"Question: {test_state['question']}")
print(f"Parent docs returned: {len(retrieved['parent_docs'])}")
print(f"Child chunks stored:  {len(retrieved['child_chunks'])}")
print(f"First parent (first 120 chars): {retrieved['parent_docs'][0][:120]}...")

## Part 13 — Generate Node

The generate node builds a prompt from the **parent docs** in state and calls the LLM. Notice that `child_chunks` never appears in this prompt — they were only used to find the right parents.

In [ ]:
# temperature=0 → deterministic outputs for reproducible demos
# Raise to 0.3-0.7 for more varied or creative responses in production
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def generate(state: ParentDocState) -> dict:
    """Generate an answer using the parent documents as context."""
    context = "\n\n".join(state["parent_docs"])
    prompt = (
        f"Answer using the provided context:\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {state['question']}"
    )
    answer = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {"answer": answer}

# Test the generate node with the retrieved state
gen_input = {**test_state, **retrieved}
gen_output = generate(gen_input)
print(f"Answer: {gen_output['answer']}")

## Part 14 — Graph Assembly

Now we wire both nodes into a `StateGraph`. The topology is the simplest possible pipeline:

```
START
  │
  ▼
retrieve_parent   ← child similarity_search finds small chunks,
  │                  ParentDocumentRetriever maps them to full parent docs
  ▼
generate          ← ChatOpenAI answers using parent doc context
  │
  ▼
 END
```

In a more advanced system you might add:
- A **reranking** node between retrieve and generate
- A **query expansion** node before retrieve
- A **grounding check** node after generate to verify the answer is supported by the retrieved docs
- A **router** that decides whether retrieval is even needed for the given question

In [ ]:
graph = StateGraph(ParentDocState)

# Register nodes
graph.add_node("retrieve_parent", retrieve_parent)
graph.add_node("generate", generate)

# Wire edges
graph.add_edge(START, "retrieve_parent")
graph.add_edge("retrieve_parent", "generate")
graph.add_edge("generate", END)

# compile() locks the topology — after this you cannot add nodes or edges
app = graph.compile()

print("Graph nodes:", list(app.get_graph().nodes.keys()))
print("Graph compiled successfully.")

## Part 15 — Full Pipeline Run

Run all three sample questions through the compiled graph. Each call is a complete cycle: embed the question → find child chunks → swap to parent docs → generate answer.

In [ ]:
results = []

for question in SAMPLE_QUESTIONS:
    print(f"\n{'=' * 60}")
    print(f"Question: {question}")

    result: ParentDocState = app.invoke(
        {
            "question": question,
            "child_chunks": [],
            "parent_docs": [],
            "answer": "",
        }
    )

    child_count = len(result["child_chunks"])
    parent_count = len(result["parent_docs"])
    child_avg = sum(len(c) for c in result["child_chunks"]) // max(child_count, 1)
    parent_avg = sum(len(d) for d in result["parent_docs"]) // max(parent_count, 1)

    print(f"Child chunks  : {child_count}  (avg {child_avg} chars each)")
    print(f"Parent docs   : {parent_count}  (avg {parent_avg} chars each)")
    print(f"Answer        : {result['answer'][:300]}")

    results.append(result)

## Part 16 — Answer Quality Comparison

To make the benefit of parent-doc retrieval concrete, let's run a side-by-side comparison: what answer does the LLM produce when given **only child chunks** vs **full parent docs**?

In [ ]:
def answer_with_context(question: str, context_docs: list[str]) -> str:
    """Call the LLM with arbitrary context docs and return the answer."""
    context = "\n\n".join(context_docs)
    prompt = (
        f"Answer using the provided context:\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

# Use the first question for comparison
q = SAMPLE_QUESTIONS[0]
child_docs_raw = [c.page_content for c in vectorstore.similarity_search(q, k=4)]
parent_docs_raw = [d.page_content for d in retriever.invoke(q)]

answer_child = answer_with_context(q, child_docs_raw)
answer_parent = answer_with_context(q, parent_docs_raw)

print(f"Question: {q}")
print()
print(f"--- ANSWER using CHILD chunks ({sum(len(c) for c in child_docs_raw)} chars of context) ---")
print(answer_child)
print()
print(f"--- ANSWER using PARENT docs ({sum(len(p) for p in parent_docs_raw)} chars of context) ---")
print(answer_parent)

## Part 17 — Chunk Size Sensitivity

What happens when we change `CHILD_CHUNK_SIZE`? This experiment rebuilds the retriever with different child sizes and measures the impact on context returned.

**Prediction before running:**
- Smaller child = more precise embedding, fewer parent docs retrieved (tighter match)
- Larger child = more context in each child, but embedding is more diffuse

In [ ]:
def build_retriever_with_child_size(child_size: int):
    """Build a fresh retriever with a given child chunk size."""
    vs = Chroma(
        collection_name=f"test-child-{child_size}",
        embedding_function=embeddings,
    )
    ds = InMemoryStore()
    c_split = RecursiveCharacterTextSplitter(chunk_size=child_size, chunk_overlap=10)
    p_split = RecursiveCharacterTextSplitter(chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=50)
    ret = ParentDocumentRetriever(
        vectorstore=vs,
        docstore=ds,
        child_splitter=c_split,
        parent_splitter=p_split,
    )
    ret.add_documents([Document(page_content=d) for d in SAMPLE_DOCS])
    return ret, vs

test_question = SAMPLE_QUESTIONS[0]
print(f"Question: {test_question}")
print()
print(f"{'Child size':>12} {'Child count in VS':>18} {'Parents returned':>17} {'Parent chars':>13}")
print("-" * 65)

for child_size in [50, 100, 200, 400]:
    ret, vs = build_retriever_with_child_size(child_size)
    children_in_vs = vs._collection.count()
    parents = ret.invoke(test_question)
    parent_chars = sum(len(p.page_content) for p in parents)
    print(f"{child_size:>12} {children_in_vs:>18} {len(parents):>17} {parent_chars:>13,}")

---
## Exercises

Work through these to cement the concepts. Answer key is in the next cell.

---

### Exercise 1 — Add a fourth document

Add this new document to the retriever and run a relevant question against it:

```python
new_doc = """Fine-tuning adapts a pre-trained model to a specific task or domain by continuing training on a smaller, 
task-specific dataset. Unlike training from scratch, fine-tuning requires far less data and computation because the 
model already has general language understanding. Two popular techniques are full fine-tuning, which updates all 
model weights, and parameter-efficient methods like LoRA, which inject small trainable matrices while freezing 
the base model. Fine-tuning is how foundation models like GPT or BERT become specialized assistants, 
code generators, or domain-specific classifiers."""
```

Question to test: `"What is the difference between full fine-tuning and LoRA?"`

---

### Exercise 2 — Custom parent chunk size

Rebuild the retriever with `PARENT_CHUNK_SIZE = 200` (much smaller parents). Run all three questions and compare the answer quality to the `PARENT_CHUNK_SIZE = 600` baseline. When does a smaller parent size hurt? When might it help?

---

### Exercise 3 — Add a grounding check node

Add a third node `check_grounding` between `generate` and `END`. The node should check whether the answer contains at least one phrase from the parent docs. If not, set `answer` to `"Could not find a grounded answer."`. Wire the graph: `START → retrieve_parent → generate → check_grounding → END`.

In [ ]:
# =============================================================================
# ANSWER KEY
# =============================================================================

# --- Exercise 1: Add a fourth document ---

new_doc_text = """Fine-tuning adapts a pre-trained model to a specific task or domain by continuing training on a smaller, 
task-specific dataset. Unlike training from scratch, fine-tuning requires far less data and computation because the 
model already has general language understanding. Two popular techniques are full fine-tuning, which updates all 
model weights, and parameter-efficient methods like LoRA, which inject small trainable matrices while freezing 
the base model. Fine-tuning is how foundation models like GPT or BERT become specialized assistants, 
code generators, or domain-specific classifiers."""

# Add to the existing retriever (the one built in Part 6)
retriever.add_documents([Document(page_content=new_doc_text)])

q1 = "What is the difference between full fine-tuning and LoRA?"
ex1_result = app.invoke({"question": q1, "child_chunks": [], "parent_docs": [], "answer": ""})
print("Exercise 1 answer:")
print(ex1_result["answer"])
print()

# --- Exercise 2: Smaller parent chunk size ---

small_parent_ret, small_vs = build_retriever_with_child_size(100)  # reuse helper, but change parent size

# Rebuild with explicit small parent size
vs_small = Chroma(collection_name="small-parent-demo", embedding_function=embeddings)
ds_small = InMemoryStore()
small_parent_retriever = ParentDocumentRetriever(
    vectorstore=vs_small,
    docstore=ds_small,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20),
)
small_parent_retriever.add_documents([Document(page_content=d) for d in SAMPLE_DOCS])

print("Exercise 2 — parent size 200 vs 600:")
for q in SAMPLE_QUESTIONS:
    parents_200 = small_parent_retriever.invoke(q)
    parents_600 = retriever.invoke(q)
    print(f"  Q: {q[:50]}")
    print(f"     200-char parents: {len(parents_200)} docs, {sum(len(p.page_content) for p in parents_200)} chars")
    print(f"     600-char parents: {len(parents_600)} docs, {sum(len(p.page_content) for p in parents_600)} chars")
print()

# --- Exercise 3: Grounding check node ---

def check_grounding(state: ParentDocState) -> dict:
    """Verify the answer contains at least one phrase from the retrieved parent docs."""
    answer_lower = state["answer"].lower()
    # Check for overlap of 4+ word phrases from parent docs
    grounded = False
    for parent in state["parent_docs"]:
        words = parent.lower().split()
        for i in range(len(words) - 3):
            phrase = " ".join(words[i:i+4])
            if phrase in answer_lower:
                grounded = True
                break
        if grounded:
            break
    if not grounded:
        return {"answer": "Could not find a grounded answer."}
    return {}

grounded_graph = StateGraph(ParentDocState)
grounded_graph.add_node("retrieve_parent", retrieve_parent)
grounded_graph.add_node("generate", generate)
grounded_graph.add_node("check_grounding", check_grounding)
grounded_graph.add_edge(START, "retrieve_parent")
grounded_graph.add_edge("retrieve_parent", "generate")
grounded_graph.add_edge("generate", "check_grounding")
grounded_graph.add_edge("check_grounding", END)
grounded_app = grounded_graph.compile()

print("Exercise 3 — grounding check:")
ex3_result = grounded_app.invoke(
    {"question": SAMPLE_QUESTIONS[0], "child_chunks": [], "parent_docs": [], "answer": ""}
)
print(f"Answer: {ex3_result['answer'][:300]}")

---
## Workshop Complete

### What you built

- A **`ParentDocumentRetriever`** that indexes small child chunks in Chroma and stores full parent docs in an `InMemoryStore`
- A **two-node LangGraph pipeline**: `retrieve_parent → generate`
- Side-by-side evidence that parent-doc context produces better LLM answers than raw child chunks
- A chunk-size sensitivity experiment showing how `CHILD_CHUNK_SIZE` and `PARENT_CHUNK_SIZE` interact

### Key takeaways

| Concept | Summary |
|---------|--------|
| Precision-vs-context tradeoff | Small chunks embed precisely; large chunks provide context |
| Two-store architecture | Vectorstore for index, docstore for lookup — different jobs |
| The swap mechanism | Child wins similarity race → parent_id lookup → parent returned |
| Chunk size ratio | Child should be 5-10x smaller than parent for best results |
| LangGraph state | `TypedDict` carries both child chunks (inspection) and parent docs (LLM) |

### Production considerations

- Replace `InMemoryStore` with a persistent docstore (Redis, Postgres) for durability
- Persist Chroma to disk or use a hosted vector DB for the embedding index
- Add a reranking step (e.g. Cohere Rerank) between retrieve and generate
- Monitor retrieval quality with a grounding score or faithfulness metric

---

**Next: example 72 — Hypothetical Document Embeddings (HyDE)**

HyDE generates a hypothetical answer to the query, embeds that answer instead of the query itself, and uses the hypothetical-answer embedding for retrieval. It's a complementary technique to Parent Document Retriever — together they address different failure modes of naive RAG.